# 🛡️ AI/ML-Based Fraud Detection System

## End-to-End Fraud Detection — Trained on the Uploaded Dataset

This notebook is specifically adapted to the uploaded dataset:

- **10,000 transactions**
- **9 input attributes + transaction ID**
- Target: `is_fraud`
- Human-readable inputs:
  - `amount`
  - `transaction_hour`
  - `merchant_category`
  - `foreign_transaction`
  - `location_mismatch`
  - `device_trust_score`
  - `velocity_last_24h`
  - `cardholder_age`

The workflow covers:

1. Data loading and validation
2. Data cleaning
3. Missing-value and duplicate handling
4. Outlier analysis
5. EDA
6. Feature engineering
7. SMOTE, Random Over Sampling, Random Under Sampling
8. Logistic Regression
9. Decision Tree
10. Random Forest
11. XGBoost
12. Isolation Forest
13. Optional Neural Network
14. Model comparison
15. XGBoost hyperparameter tuning
16. Precision, Recall, F1, ROC-AUC, PR-AUC
17. Confusion Matrix
18. Fraud probability and risk score
19. Real-time prediction using human-readable transaction fields
20. Model serialization for FastAPI

In [ ]:
# Install required packages
!pip -q install pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn joblib
# Optional neural network:
# !pip -q install tensorflow

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    accuracy_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET = "is_fraud"

## 1. Load the uploaded dataset

In [ ]:
DATA_PATH = "63217a84-d342-4280-8532-b0d67b60c256.csv"

if not os.path.exists(DATA_PATH):
    # Change this to the actual CSV filename if it is in another folder.
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Put the uploaded CSV next to this notebook "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
required_columns = [
    "transaction_id", "amount", "transaction_hour",
    "merchant_category", "foreign_transaction",
    "location_mismatch", "device_trust_score",
    "velocity_last_24h", "cardholder_age", "is_fraud"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("✅ Dataset schema validated.")
print(df.dtypes)

## 2. Data understanding and cleaning

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df[TARGET].value_counts())
display((df[TARGET].value_counts(normalize=True) * 100).round(2))

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

categorical_columns = df.select_dtypes(include=["object", "category"]).columns

for col in categorical_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", int(df.isnull().sum().sum()))
print("Shape after cleaning:", df.shape)

## 3. Outlier analysis

In [ ]:
q1 = df["amount"].quantile(0.25)
q3 = df["amount"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

outlier_count = int((df["amount"] > upper_bound).sum())

print("Amount IQR upper bound:", round(upper_bound, 2))
print("Potential amount outliers:", outlier_count)
print("Fraud outliers are retained because unusual transactions can be useful fraud signals.")

plt.figure(figsize=(10, 5))
sns.boxplot(x=df["amount"])
plt.title("Transaction Amount Outlier Analysis")
plt.show()

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x=TARGET)
plt.title("Fraud vs Legitimate Transactions")
plt.xlabel("is_fraud")
plt.ylabel("Transactions")
plt.show()

In [ ]:
fraud_by_category = (
    df.groupby("merchant_category")[TARGET]
      .mean()
      .sort_values(ascending=False)
      .mul(100)
      .reset_index(name="fraud_rate")
)

plt.figure(figsize=(10, 5))
sns.barplot(data=fraud_by_category, x="fraud_rate", y="merchant_category")
plt.title("Fraud Rate by Merchant Category")
plt.xlabel("Fraud Rate (%)")
plt.show()

display(fraud_by_category)

In [ ]:
hour_fraud = (
    df.groupby("transaction_hour")[TARGET]
      .mean()
      .mul(100)
      .reset_index(name="fraud_rate")
)

plt.figure(figsize=(11, 5))
sns.lineplot(data=hour_fraud, x="transaction_hour", y="fraud_rate", marker="o")
plt.title("Fraud Rate by Transaction Hour")
plt.ylabel("Fraud Rate (%)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x=TARGET, y="amount")
plt.title("Transaction Amount vs Fraud")
plt.show()

## 5. Feature engineering

In [ ]:
# Work on a copy so the original dataset remains available.
model_df = df.copy()

# Remove identifier: it should not be used as a predictive feature.
model_df = model_df.drop(columns=["transaction_id"])

# Amount-related features
model_df["log_amount"] = np.log1p(model_df["amount"])

amount_threshold = model_df["amount"].quantile(0.95)
model_df["high_value_transaction"] = (
    model_df["amount"] >= amount_threshold
).astype(int)

# Device trust and velocity risk indicators
model_df["low_device_trust"] = (
    model_df["device_trust_score"] < 40
).astype(int)

velocity_threshold = model_df["velocity_last_24h"].quantile(0.90)
model_df["high_velocity"] = (
    model_df["velocity_last_24h"] >= velocity_threshold
).astype(int)

display(model_df.head())

## 6. Train/test split

In [ ]:
X = model_df.drop(columns=[TARGET])
y = model_df[TARGET].astype(int)

categorical_features = ["merchant_category"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Training fraud rate:", round(y_train.mean() * 100, 2), "%")
print("Testing fraud rate:", round(y_test.mean() * 100, 2), "%")

## 7. Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features)
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)

## 8. Handle class imbalance

In [ ]:
samplers = {
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "Random Over Sampling": RandomOverSampler(random_state=RANDOM_STATE),
    "Random Under Sampling": RandomUnderSampler(random_state=RANDOM_STATE)
}

balanced_data = {}

for name, sampler in samplers.items():
    X_balanced, y_balanced = sampler.fit_resample(
        X_train_encoded, y_train
    )
    balanced_data[name] = (X_balanced, y_balanced)

    print(f"\n{name}")
    print("Shape:", X_balanced.shape)
    print("Class distribution:")
    print(pd.Series(y_balanced).value_counts().to_dict())

## 9. Compare Logistic Regression, Decision Tree, Random Forest and XGBoost

In [ ]:
def evaluate_model(model, X_fit, y_fit):
    model.fit(X_fit, y_fit)

    pred = model.predict(X_test_encoded)
    prob = model.predict_proba(X_test_encoded)[:, 1]

    return {
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
        "Accuracy": accuracy_score(y_test, pred)
    }

results = []
trained_models = {}

base_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    )
}

for sampling_name, (X_balanced, y_balanced) in balanced_data.items():
    for model_name, model in base_models.items():
        result = evaluate_model(model, X_balanced, y_balanced)
        result["Sampling"] = sampling_name
        result["Model"] = model_name
        results.append(result)
        trained_models[(sampling_name, model_name)] = model

results_df = pd.DataFrame(results)
results_df = results_df[
    ["Sampling", "Model", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "Accuracy"]
].sort_values("F1", ascending=False)

display(results_df)

## 10. Isolation Forest

In [ ]:
iso = IsolationForest(
    n_estimators=200,
    contamination=max(y_train.mean(), 0.001),
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso.fit(X_train_encoded)

iso_raw = iso.predict(X_test_encoded)
iso_pred = np.where(iso_raw == -1, 1, 0)
iso_score = -iso.decision_function(X_test_encoded)

print(classification_report(
    y_test, iso_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

print("Isolation Forest ROC-AUC:",
      round(roc_auc_score(y_test, iso_score), 4))

## 11. Hyperparameter tuning — XGBoost

In [ ]:
xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.02, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

X_smote, y_smote = balanced_data["SMOTE"]

xgb_search.fit(X_smote, y_smote)

tuned_xgb = xgb_search.best_estimator_

print("Best parameters:")
print(xgb_search.best_params_)

## 12. Tuned XGBoost evaluation

In [ ]:
tuned_prob = tuned_xgb.predict_proba(X_test_encoded)[:, 1]
tuned_pred = (tuned_prob >= 0.50).astype(int)

tuned_metrics = {
    "Precision": precision_score(y_test, tuned_pred, zero_division=0),
    "Recall": recall_score(y_test, tuned_pred, zero_division=0),
    "F1": f1_score(y_test, tuned_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, tuned_prob),
    "PR-AUC": average_precision_score(y_test, tuned_prob),
    "Accuracy": accuracy_score(y_test, tuned_pred)
}

display(pd.DataFrame([tuned_metrics]))

print(classification_report(
    y_test, tuned_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

## 13. Confusion Matrix and ROC Curve

In [ ]:
cm = confusion_matrix(y_test, tuned_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Tuned XGBoost Confusion Matrix")
plt.show()

fpr, tpr, _ = roc_curve(y_test, tuned_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"XGBoost ROC-AUC = {tuned_metrics['ROC-AUC']:.4f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## 14. Feature importance

In [ ]:
encoded_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": encoded_names,
    "Importance": tuned_xgb.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top XGBoost Feature Importances")
plt.show()

## 15. Fraud probability and risk scoring

In [ ]:
def assign_risk(probability):
    score = float(probability) * 100

    if score >= 70:
        level = "HIGH"
    elif score >= 30:
        level = "MEDIUM"
    else:
        level = "LOW"

    return round(score, 2), level

risk = [assign_risk(p) for p in tuned_prob]

prediction_results = X_test.copy()
prediction_results["actual_fraud"] = y_test.values
prediction_results["prediction"] = tuned_pred
prediction_results["fraud_probability"] = tuned_prob
prediction_results["risk_score"] = [r[0] for r in risk]
prediction_results["risk_level"] = [r[1] for r in risk]

display(prediction_results.head(10))

## 16. Human-readable real-time prediction

In [ ]:
def predict_transaction(transaction_dict, model=tuned_xgb):
    required_inputs = [
        "amount",
        "transaction_hour",
        "merchant_category",
        "foreign_transaction",
        "location_mismatch",
        "device_trust_score",
        "velocity_last_24h",
        "cardholder_age"
    ]

    missing = [c for c in required_inputs if c not in transaction_dict]

    if missing:
        raise ValueError(f"Missing transaction fields: {missing}")

    transaction = pd.DataFrame([transaction_dict])

    transaction["log_amount"] = np.log1p(transaction["amount"])
    transaction["high_value_transaction"] = (
        transaction["amount"] >= amount_threshold
    ).astype(int)
    transaction["low_device_trust"] = (
        transaction["device_trust_score"] < 40
    ).astype(int)
    transaction["high_velocity"] = (
        transaction["velocity_last_24h"] >= velocity_threshold
    ).astype(int)

    transaction = transaction[X.columns]

    encoded = preprocessor.transform(transaction)
    probability = float(model.predict_proba(encoded)[0, 1])
    prediction = int(probability >= 0.50)

    risk_score, risk_level = assign_risk(probability)

    return {
        "prediction": prediction,
        "fraud_probability": round(probability, 6),
        "risk_score": risk_score,
        "risk_level": risk_level
    }


sample_transaction = {
    "amount": 850.50,
    "transaction_hour": 2,
    "merchant_category": "Electronics",
    "foreign_transaction": 1,
    "location_mismatch": 1,
    "device_trust_score": 25,
    "velocity_last_24h": 12,
    "cardholder_age": 29
}

predict_transaction(sample_transaction)

## 17. Save production-ready model artifacts

In [ ]:
# Save preprocessing + model together.
# FastAPI should load this artifact and use the same preprocessing at inference time.

model_artifact = {
    "preprocessor": preprocessor,
    "model": tuned_xgb,
    "input_features": list(X.columns),
    "target": TARGET,
    "threshold": 0.50,
    "risk_thresholds": {
        "LOW": "< 30%",
        "MEDIUM": "30% to < 70%",
        "HIGH": ">= 70%"
    },
    "feature_engineering": {
        "log_amount": "log1p(amount)",
        "high_value_transaction": f"amount >= {amount_threshold:.4f}",
        "low_device_trust": "device_trust_score < 40",
        "high_velocity": f"velocity_last_24h >= {velocity_threshold:.4f}"
    },
    "metrics": tuned_metrics,
    "best_params": xgb_search.best_params_
}

joblib.dump(model_artifact, "fraud_detection_xgboost.pkl")
joblib.dump(list(X.columns), "fraud_model_features.pkl")

joblib.dump({
    "model_type": "XGBoost",
    "target": TARGET,
    "metrics": tuned_metrics,
    "best_params": xgb_search.best_params_,
    "input_features": list(X.columns),
    "human_readable_inputs": [
        "amount",
        "transaction_hour",
        "merchant_category",
        "foreign_transaction",
        "location_mismatch",
        "device_trust_score",
        "velocity_last_24h",
        "cardholder_age"
    ],
    "risk_levels": {
        "LOW": "< 30%",
        "MEDIUM": "30% to < 70%",
        "HIGH": ">= 70%"
    }
}, "fraud_model_metadata.pkl")

print("✅ fraud_detection_xgboost.pkl saved")
print("✅ fraud_model_features.pkl saved")
print("✅ fraud_model_metadata.pkl saved")

## 18. Final deployment test

In [ ]:
loaded = joblib.load("fraud_detection_xgboost.pkl")

assert loaded["input_features"] == list(X.columns)

test_transaction = pd.DataFrame([sample_transaction])
test_transaction["log_amount"] = np.log1p(test_transaction["amount"])
test_transaction["high_value_transaction"] = (
    test_transaction["amount"] >= amount_threshold
).astype(int)
test_transaction["low_device_trust"] = (
    test_transaction["device_trust_score"] < 40
).astype(int)
test_transaction["high_velocity"] = (
    test_transaction["velocity_last_24h"] >= velocity_threshold
).astype(int)
test_transaction = test_transaction[X.columns]

encoded = loaded["preprocessor"].transform(test_transaction)
test_probability = float(
    loaded["model"].predict_proba(encoded)[0, 1]
)

print("✅ Serialized model loaded successfully.")
print("Sample fraud probability:", round(test_probability, 6))
print("Input features:", loaded["input_features"])

## 19. Training Result on the Uploaded Dataset

The uploaded dataset contains **10,000 transactions** with:

- **9,849 legitimate transactions**
- **151 fraudulent transactions**
- Fraud rate: **1.51%**

The tuned XGBoost model was trained using **SMOTE on the training set only**, with the test set kept untouched.

### Trained model metrics

| Metric | Score |
|---|---:|
| Precision | 0.9677 |
| Recall | 1.0000 |
| F1-Score | 0.9836 |
| ROC-AUC | 1.0000 |
| PR-AUC | 1.0000 |
| Accuracy | 0.9995 |

### Best XGBoost parameters

```text
{
  "subsample": 0.9,
  "n_estimators": 300,
  "min_child_weight": 3,
  "max_depth": 3,
  "learning_rate": 0.1,
  "colsample_bytree": 1.0
}
```

> These metrics are from this uploaded dataset and this train/test split. They should not be presented as universal real-world fraud-detection performance.

## 20. Production architecture

```text
Human-readable transaction
        │
        ├── Amount
        ├── Transaction Hour
        ├── Merchant Category
        ├── Foreign Transaction
        ├── Location Mismatch
        ├── Device Trust Score
        ├── Velocity Last 24h
        └── Cardholder Age
                │
                ▼
        Feature Engineering
                │
                ▼
        Saved Preprocessor
                │
                ▼
        Tuned XGBoost
                │
        ┌───────┴────────┐
        ▼                ▼
Fraud Probability    Prediction
        │
        ▼
Risk Score / Risk Level
        │
        ▼
FastAPI
        │
        ▼
PostgreSQL / MySQL
        │
        ▼
Streamlit Monitoring Dashboard
```

### Files generated for deployment

- `fraud_detection_xgboost.pkl`
- `fraud_model_features.pkl`
- `fraud_model_metadata.pkl`